### Character-Level Name Generator

Predicts the next character in a name given the previous few characters, using a small neural network with a learned embedding layer, trained on ~32,000 real first names. Sampling from the trained model repeatedly generates entirely new, plausible-sounding names.

In [ ]:
import urllib.request

url = 'https://raw.githubusercontent.com/karpathy/makemore/master/names.txt'
urllib.request.urlretrieve(url, 'names.txt')

words = open('names.txt', 'r').read().splitlines()

In [ ]:
import numpy as np

In [ ]:
all_text = ''.join(words)
unique_chars = set(all_text)
sorted_chars = sorted(unique_chars)
print(sorted_chars)

In [ ]:
#converting the string to integer, only to process the data for our network
#'.' is used as a marker to separate two words
stoi = {ch: i+1 for i, ch in enumerate(sorted_chars)}
stoi['.'] = 0

In [ ]:
#converting the integer back to string
itos = {i: ch for ch, i in stoi.items()}

In [ ]:
context_size = 3

X = []   #contexts
Y = []   #target next integer

for w in words:
    context = [0] * context_size
    for ch in w + '.':
        ix = stoi[ch]
        X.append(context)
        Y.append(ix)
        context = context[1:] + [ix]

X = np.array(X)
Y = np.array(Y)


In [ ]:
embedding_dim = 10
C = np.random.randn(27, embedding_dim)

In [ ]:
#initiliasation of weights and biases
input_size = 30
hidden_size = 100
output_size = 27

W1 = np.random.randn(input_size,hidden_size) * np.sqrt(2 / input_size)
b1 = np.zeros((1,hidden_size))
W2 = np.random.randn(hidden_size,output_size) * np.sqrt(2 / hidden_size)
b2 = np.zeros((1,output_size))

In [ ]:
#ReLU
def ReLU(x):
    return np.maximum(0, x)
#Softmax
def soft_max(x):
    x = x - np.max(x, axis=1, keepdims=True)
    exp_x = np.exp(x)
    sum_exp = np.sum(exp_x, axis=1, keepdims=True)
    return exp_x / sum_exp

In [ ]:
#forward_pass
def forward_pass(X,C,W1,b1,W2,b2):
    emb = C[X]
    emb_flat = emb.reshape(emb.shape[0], -1)

    Z1 = emb_flat @ W1 + b1
    A1 = ReLU(Z1)
    Z2 = A1 @ W2 + b2
    A2 = soft_max(Z2)
    return emb_flat,Z1,A1,Z2,A2

In [ ]:
emb_flat, Z1, A1, Z2, A2 = forward_pass(X[:5], C, W1, b1, W2, b2)
print(A2.shape)

In [ ]:
#Loss function
def cross_entropy_loss(A2, y_onehot):
    correct_probs = np.sum(y_onehot * A2, axis=1)
    loss = np.mean(-np.log(correct_probs + 1e-9))
    return loss

In [ ]:
#Backward Pass
def backward_pass(X_batch, y_onehot, emb_flat, Z1, A1, A2, W1, W2, C, context_size, embedding_dim):
    batch_size = X_batch.shape[0]
    
    # output layer gradient (softmax + cross-entropy shortcut)
    delta_Z2 = A2 - y_onehot
    
    # gradients for W2, b2
    delta_W2 = A1.T @ delta_Z2 / batch_size
    delta_b2 = np.mean(delta_Z2, axis=0, keepdims=True)
    
    # propagate to A1, then through ReLU
    delta_A1 = delta_Z2 @ W2.T
    delta_Z1 = delta_A1 * (Z1 > 0)
    
    # gradients for W1, b1
    delta_W1 = emb_flat.T @ delta_Z1 / batch_size
    delta_b1 = np.mean(delta_Z1, axis=0, keepdims=True)
    
    # propagate back to embeddings
    delta_emb_flat = delta_Z1 @ W1.T
    delta_emb = delta_emb_flat.reshape(batch_size, context_size, embedding_dim)
    
    # accumulate gradient into embedding table C
    delta_C = np.zeros_like(C)
    np.add.at(delta_C, X_batch, delta_emb)
    
    return delta_W1, delta_b1, delta_W2, delta_b2, delta_C

In [ ]:
#Training Loop
learning_rate = 0.01
epochs = 30
batch_size = 64
context_size = 3
embedding_dim = 10

loss_history = []

for epoch in range(epochs):
    indices = np.random.permutation(X.shape[0])
    X_shuffled = X[indices]
    Y_shuffled = Y[indices]
    
    num_batches = X_shuffled.shape[0] // batch_size
    
    for i in range(num_batches):
        start = i * batch_size
        end = start + batch_size
        X_batch = X_shuffled[start:end]
        Y_batch = Y_shuffled[start:end]
        y_onehot_batch = np.eye(27)[Y_batch]
        
        emb_flat, Z1, A1, Z2, A2 = forward_pass(X_batch, C, W1, b1, W2, b2)
        
        delta_W1, delta_b1, delta_W2, delta_b2, delta_C = backward_pass(
            X_batch, y_onehot_batch, emb_flat, Z1, A1, A2, W1, W2, C, context_size, embedding_dim
        )
        
        W1 = W1 - learning_rate * delta_W1
        b1 = b1 - learning_rate * delta_b1
        W2 = W2 - learning_rate * delta_W2
        b2 = b2 - learning_rate * delta_b2
        C  = C  - learning_rate * delta_C
    
    y_onehot_full = np.eye(27)[Y]
    _, _, _, _, A2_full = forward_pass(X, C, W1, b1, W2, b2)
    epoch_loss = cross_entropy_loss(A2_full, y_onehot_full)
    loss_history.append(epoch_loss)
    print(f"Epoch {epoch+1}/{epochs} - Loss: {epoch_loss:.4f}")

In [ ]:
#Generating some names 
def generate_name(C, W1, b1, W2, b2, context_size=3, max_len=20):
    context = [0] * context_size
    generated = []
    
    for _ in range(max_len):
        X_input = np.array([context])
        _, _, _, _, A2 = forward_pass(X_input, C, W1, b1, W2, b2)
        probs = A2[0]
        
        ix = np.random.choice(len(probs), p=probs)
        
        if ix == 0:
            break
        
        generated.append(itos[ix])
        context = context[1:] + [ix]
    
    return ''.join(generated)

In [ ]:
for _ in range(10):
    print(generate_name(C, W1, b1, W2, b2))